# S4 · AndinaLog 03B · Notebook 1 · Diagnóstico de calidad IoT

**Objetivo:** diagnosticar el dataset Bronze sin transformar sus diez columnas originales. Este notebook orquesta `quality_engine.py` y el catálogo externo `catalogo_reglas_iot.json`. Las conversiones utilizadas para comprobar reglas son temporales. No hay EDA ni tratamiento Silver.

La clave candidata es `viaje_id + timestamp`; requiere validación de negocio. Una copia idéntica se marca desde su segunda aparición. Si la misma clave reúne contenidos distintos, se marcan todas las filas involucradas. `F` es una advertencia pendiente de estandarización, sin cuarentena por sí sola. Las reglas de integridad se ejecutan solo si se proporcionan datasets de referencia reales; el reporte deja constancia de las no ejecutadas.

Convención temporal: los timestamps Bronze sin zona se interpretan como hora de Bolivia (America/La_Paz) para el tratamiento posterior; UTC será la hora canónica procesada. El diagnóstico conserva el texto original. El docente confirmó que las fechas sin zona del caso se interpretan como hora de Bolivia. El diagnóstico conserva el texto Bronze; el tratamiento genera UTC. K es unidad operacional no esperada.

La entrega v7 contiene solo `diagnosticado` y `cuarentena`. Cada fila de `diagnosticado` incluye `incidencias_json` con reglas, columnas, severidad, valor y explicación; `cuarentena` es el subconjunto señalado por la política diagnóstica.


## 1 · Configuración y origen

Coloca este notebook, `quality_engine.py` y `catalogo_reglas_iot.json` en la misma carpeta. En local, ejecútalo desde cualquier carpeta del proyecto que contenga `datasets/AndinaLog_03B_Bronce/`. En Colab, monta Drive y ajusta la ruta. El Bronze se lee como texto y nunca se sobrescribe.


In [ ]:
from pathlib import Path
import hashlib
import os
import sys
import tempfile
import pandas as pd
# quality_engine.py está en la carpeta superior, compartido por los CSV.
CARPETA_NOTEBOOK = Path.cwd()
if not (CARPETA_NOTEBOOK / "catalogo_reglas_iot.json").is_file():
    CARPETA_NOTEBOOK = Path("proyecto-integrador/diagnostico/andinalog_iot_telemetry")
sys.path.insert(0, str(CARPETA_NOTEBOOK.parent.resolve()))
from quality_engine import cargar_catalogo, diagnosticar, reportes

ENTORNO = "auto"  # auto, local, drive
RUTA_PROYECTO_DRIVE = "/content/drive/MyDrive/GIAD"
CARPETA_DATASETS = "AndinaLog_03B_Bronce"
NOMBRE_CSV = "andinalog_iot_telemetry.csv"
VERSION_DIAGNOSTICO = "GIAD-M3-S4-IOT-diagnostico-v7"
RUTA_CATALOGO = CARPETA_NOTEBOOK / "catalogo_reglas_iot.json"

# Indica rutas reales solamente cuando estén disponibles. No se infieren fuentes.
RUTAS_REFERENCIA = {
    "productos": "proyecto-integrador/andinalog_productos/notebook2/salidas/andinalog_productos_silver.csv",
    "flota": "proyecto-integrador/andinalog_flota/notebook2/salidas/andinalog_flota_silver.csv",
    # No se entregó maestro de viajes; su regla quedará NO_EJECUTADA.
}

def encontrar_raiz_local():
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        if (carpeta / "datasets" / CARPETA_DATASETS / NOMBRE_CSV).is_file():
            return carpeta
    raise FileNotFoundError("No se encontró el CSV Bronze en la carpeta actual o sus padres")

def configurar_rutas(entorno):
    if entorno == "auto":
        entorno = "drive" if "google.colab" in sys.modules else "local"
    if entorno == "drive":
        from google.colab import drive
        drive.mount("/content/drive")
        raiz = Path(RUTA_PROYECTO_DRIVE)
    elif entorno == "local":
        raiz = encontrar_raiz_local()
    else:
        raise ValueError("ENTORNO debe ser auto, local o drive")
    bronze = raiz / "datasets" / CARPETA_DATASETS / NOMBRE_CSV
    salidas = raiz / "proyecto-integrador" / "diagnostico"/ "andinalog_iot_telemetry" / "salidas"
    if not bronze.is_file():
        raise FileNotFoundError(bronze)
    return bronze, salidas

catalogo = cargar_catalogo(RUTA_CATALOGO)
RUTA_BRONZE, DIRECTORIO_SALIDAS = configurar_rutas(ENTORNO)
print("Bronze:", RUTA_BRONZE)
print("Catálogo:", RUTA_CATALOGO, catalogo["version_catalogo"])


Bronze: c:\Users\remrodri\Github\practicasNotebookColab\datasets\AndinaLog_03B_Bronce\andinalog_iot_telemetry.csv
Catálogo: catalogo_reglas_iot.json 1.0.0


## 2 · Carga y diagnóstico

`fila_bronze` cuenta desde 1 y conserva la posición de la fila de datos en el CSV. Cada incumplimiento genera una incidencia con regla, dimensión, severidad, acción y valor original. Una fila puede tener varias incidencias.


In [2]:
HASH_BRONZE = hashlib.sha256(RUTA_BRONZE.read_bytes()).hexdigest()
df_bronze = pd.read_csv(RUTA_BRONZE, dtype="string", encoding="utf-8-sig", keep_default_na=False)
referencias = {}
for nombre, ruta in RUTAS_REFERENCIA.items():
    ruta = RUTA_BRONZE.parents[2] / ruta if not Path(ruta).is_absolute() else Path(ruta)
    if not ruta.is_file():
        raise FileNotFoundError(f"Referencia declarada y ausente: {ruta}")
    referencias[nombre] = pd.read_csv(ruta, dtype="string", encoding="utf-8-sig", keep_default_na=False)

df_diagnosticado, df_problemas, estado_reglas = diagnosticar(df_bronze, catalogo, referencias)
df_problemas["version_diagnostico"] = VERSION_DIAGNOSTICO
df_diagnosticado["version_diagnostico"] = VERSION_DIAGNOSTICO
df_diagnosticado["version_catalogo"] = catalogo["version_catalogo"]
df_diagnosticado["sha256_bronze"] = HASH_BRONZE
df_diagnosticado["zona_horaria_origen"] = catalogo["zona_horaria_origen"]
df_diagnosticado["reglas_no_ejecutadas"] = "|".join(estado_reglas.loc[estado_reglas["estado"].eq("NO_EJECUTADA"), "rule_id"])
df_cuarentena = df_diagnosticado.loc[df_diagnosticado["en_cuarentena"]].copy()
display(estado_reglas)
display(df_problemas.head())


,rule_id,estado,motivo,incidencias
0,IOT-R001,EJECUTADA,,0
1,IOT-R002,EJECUTADA,,15
2,IOT-R003,EJECUTADA,,0
3,IOT-R004,EJECUTADA,,0
4,IOT-R005,EJECUTADA,,0
5,IOT-R006,EJECUTADA,,0
6,IOT-R007,EJECUTADA,,0
7,IOT-R008,EJECUTADA,,50
8,IOT-R009,EJECUTADA,,0
9,IOT-R010,EJECUTADA,,0


,fila_bronze,rule_id,dimension_calidad,severidad,accion,tipo_problema,columna_afectada,codigo_error,valor_original,detalle,version_catalogo,version_diagnostico
0,8,IOT-R019,COMPLETITUD,CRITICAL,CUARENTENA,FALTANTE,,FALTANTE,,Humedad vacía,1.0.0,GIAD-M3-S4-IOT-diagnostico-v5
1,121,IOT-R019,COMPLETITUD,CRITICAL,CUARENTENA,FALTANTE,,FALTANTE,,Humedad vacía,1.0.0,GIAD-M3-S4-IOT-diagnostico-v5
2,125,IOT-R015,CONFORMIDAD,WARNING,REVISAR,CONFORMIDAD,,PENDIENTE_CONVERSION_F,F,Lectura válida en F; estandarización correspon...,1.0.0,GIAD-M3-S4-IOT-diagnostico-v5
3,146,IOT-R008,CONFORMIDAD,CRITICAL,CUARENTENA,FORMATO,,FORMATO_INVALIDO,cam-19,Formato esperado CAM-00,1.0.0,GIAD-M3-S4-IOT-diagnostico-v5
4,213,IOT-R019,COMPLETITUD,CRITICAL,CUARENTENA,FALTANTE,,FALTANTE,,Humedad vacía,1.0.0,GIAD-M3-S4-IOT-diagnostico-v5


## 3 · Métricas y reportes de calidad

Los porcentajes de cada agrupación usan el total de filas Bronze como denominador. Las incidencias pueden superar las filas afectadas. Las reglas de integridad sin fuente disponible aparecen como `NO_EJECUTADA`, no como aprobadas.


In [3]:
tablas_reporte = reportes(df_diagnosticado, df_problemas, estado_reglas, catalogo, RUTA_BRONZE.name, HASH_BRONZE)
reporte_calidad = tablas_reporte["metricas"]
reporte_calidad.loc[len(reporte_calidad)] = ["version_diagnostico", VERSION_DIAGNOSTICO]
for nombre in ["metricas", "por_regla", "por_columna", "por_dimension"]:
    print(nombre)
    display(tablas_reporte[nombre])


metricas


,metrica,valor
0,archivo_bronze,andinalog_iot_telemetry.csv
1,sha256_bronze,edb7afe2f7fb836e59fe605d30c88b3b5b13a6d8ab2ec0...
2,version_catalogo,1.0.0
3,filas_bronze,28920
4,filas_con_problemas,554
5,filas_en_cuarentena,505
6,filas_solo_warning,49
7,incidencias,560
8,incidencias_critical,510
9,incidencias_warning,50


por_regla


,rule_id,dimension_calidad,severidad,accion,tipo_problema,columna_afectada,codigo_error,incidencias,filas_afectadas,porcentaje_filas
0,IOT-R002,CONFORMIDAD,CRITICAL,CUARENTENA,FORMATO,,FECHA_INVALIDA,15,15,0.05
1,IOT-R008,CONFORMIDAD,CRITICAL,CUARENTENA,FORMATO,,FORMATO_INVALIDO,50,50,0.17
2,IOT-R011,UNICIDAD,CRITICAL,CUARENTENA,UNICIDAD,,DUPLICADO_IDENTICO,115,115,0.40
3,IOT-R012,UNICIDAD,CRITICAL,CUARENTENA,UNICIDAD,,CLAVE_EN_CONFLICTO,10,10,0.03
4,IOT-R014,VALIDEZ,CRITICAL,CUARENTENA,VALIDEZ,,UNIDAD_NO_RECONOCIDA,5,5,0.02
5,IOT-R015,CONFORMIDAD,WARNING,REVISAR,CONFORMIDAD,,PENDIENTE_CONVERSION_F,50,50,0.17
6,IOT-R016,COMPLETITUD,CRITICAL,CUARENTENA,FALTANTE,,FALTANTE,80,80,0.28
7,IOT-R018,VALIDEZ,CRITICAL,CUARENTENA,VALIDEZ,,VALOR_CENTINELA,120,120,0.41
8,IOT-R019,COMPLETITUD,CRITICAL,CUARENTENA,FALTANTE,,FALTANTE,100,100,0.35
9,IOT-R021,VALIDEZ,CRITICAL,CUARENTENA,VALIDEZ,,FUERA_RANGO,15,15,0.05


por_columna


,columna_afectada,incidencias,filas_afectadas,porcentaje_filas
0,,560,554,1.92


por_dimension


,dimension_calidad,incidencias,filas_afectadas,porcentaje_filas
0,COMPLETITUD,180,180,0.62
1,CONFORMIDAD,115,114,0.39
2,UNICIDAD,125,125,0.43
3,VALIDEZ,140,140,0.48


## 4 · Comprobaciones y exportación

Solo se exportan `diagnosticado` y `cuarentena`. Los detalles de las incidencias viajan dentro de `incidencias_json` y los contadores se concilian con esa lista. Bronze no se sobrescribe.


In [4]:
pd.testing.assert_frame_equal(df_diagnosticado[catalogo["columnas_bronze"]], df_bronze)
assert len(df_diagnosticado) == len(df_bronze)
assert df_diagnosticado["fila_bronze"].is_unique
assert (df_diagnosticado["cantidad_problemas"] == df_diagnosticado["cantidad_critical"] + df_diagnosticado["cantidad_warning"]).all()
assert set(df_cuarentena["fila_bronze"]) == set(df_problemas.loc[df_problemas["accion"].eq("CUARENTENA"), "fila_bronze"])
assert df_problemas["rule_id"].isin([r["rule_id"] for r in catalogo["reglas"]]).all()
assert df_diagnosticado["incidencias_json"].map(lambda s: len(__import__("json").loads(s))).eq(df_diagnosticado["cantidad_problemas"]).all()

def exportar_salidas(directorio, tablas, ruta_bronze, huella_inicial):
    if hashlib.sha256(ruta_bronze.read_bytes()).hexdigest() != huella_inicial:
        raise RuntimeError("El CSV Bronze cambió durante la ejecución")
    directorio.mkdir(parents=True, exist_ok=True)
    temporales = {}
    try:
        for nombre, tabla in tablas.items():
            destino = directorio / nombre
            with tempfile.NamedTemporaryFile(mode="w", suffix=".csv", prefix=".tmp_iot_", dir=directorio,
                                             encoding="utf-8-sig", newline="", delete=False) as tmp:
                tabla.to_csv(tmp, index=False)
                temporales[destino] = Path(tmp.name)
        for destino, temporal in temporales.items():
            os.replace(temporal, destino)
    finally:
        for temporal in temporales.values():
            temporal.unlink(missing_ok=True)
    return list(temporales)

prefijo = "andinalog_iot_telemetry_v7_"
tablas_salida = {
    prefijo + "diagnosticado.csv": df_diagnosticado,
    prefijo + "cuarentena.csv": df_cuarentena,
}
rutas_creadas = exportar_salidas(DIRECTORIO_SALIDAS, tablas_salida, RUTA_BRONZE, HASH_BRONZE)
for ruta in rutas_creadas:
    print(ruta)
print("Bronze intacto; dos archivos de diagnóstico exportados")


c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\diagnostico\salidas\andinalog_iot_telemetry_diagnosticado.csv
c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\diagnostico\salidas\andinalog_iot_telemetry_problemas.csv
c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\diagnostico\salidas\andinalog_iot_telemetry_cuarentena.csv
c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\diagnostico\salidas\andinalog_iot_telemetry_reporte_calidad.csv
c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\diagnostico\salidas\andinalog_iot_telemetry_reporte_por_regla.csv
c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\diagnostico\salidas\andinalog_iot_telemetry_reporte_por_columna.csv
c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\diagnostico\salidas\andinalog_iot_telemetry_reporte_por_dimension.csv
c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\diagnosti

## Alcance y siguiente etapa

Las banderas térmicas se validan solo como 0/1. La coherencia operativa y la ventana futura de 60 minutos requieren umbrales y definición temporal aprobados. El notebook 2 podrá consumir las incidencias y decidir el tratamiento Silver; este notebook solo diagnostica.

La coherencia del flag respecto al rango del producto se evalúa sin alterar Bronze: F se interpreta temporalmente en Celsius; K y -999 no son evaluables para esa regla. La zona horaria del origen no está verificada.

La hora de Bolivia para fechas sin zona fue confirmada por el docente; la conversión UTC corresponde al tratamiento.
